In [0]:
from pyspark.sql import Row, functions as F
import random

random.seed(42)
regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Grocery", "Clothing", "Furniture"]

rows = [Row(
    order_id=i,
    region=random.choice(regions),
    product_category=random.choice(categories),
    amount=round(random.uniform(10, 500), 2),
    order_date=f"2026-0{random.randint(1,9)}-{random.randint(10,28)}"
) for i in range(1, 201)]

sales_df = spark.createDataFrame(rows)

spark.sql("CREATE DATABASE IF NOT EXISTS retail")
spark.sql("USE retail")

sales_df.write.format("delta").mode("overwrite").saveAsTable("retail.bronze_sales")

category_df = spark.createDataFrame(
    [("Electronics", "High margin"), ("Grocery", "Low margin"),
     ("Clothing", "Seasonal"), ("Furniture", "High margin")],
    ["product_category", "notes"])

print("Setup complete.")
spark.sql("SELECT * FROM retail.bronze_sales LIMIT 5").show()

In [0]:
print(spark.version)
spark.sql("SELECT * FROM retail.bronze_sales LIMIT 3").show()

In [0]:
df = spark.read.table("retail.bronze_sales")           # data engineering
df.createOrReplaceTempView("txns")
spark.sql("SELECT region, SUM(amount) FROM txns GROUP BY region").show()

In [0]:
# SparkContext is not available on Spark Connect (serverless/standard compute)
# Display Spark version and deployment mode instead
print(f"Spark version: {spark.version}")
print(f"Deployment mode: Serverless (Spark Connect)")

In [0]:
print("Python cell")

In [0]:
%sql
SELECT current_date();

In [0]:
%sql
SELECT * FROM retail.bronze_sales LIMIT 3;

In [0]:
# spark.databricks.clusterUsageTags.sparkVersion is not available on Serverless
# Use spark.version instead
display(spark.version)

In [0]:
job_config = {
  "name": "retail_daily_etl",
  "tasks": [{
    "task_key": "run_etl",
    "notebook_task": {"notebook_path": "/Workspace/Users/you/spark-fundamentals"},
    "new_cluster": {"num_workers": 2, "spark_version": "14.3.x-scala2.12", "node_type_id": "i3.xlarge"}
  }]
}
print(job_config)

In [0]:
df = spark.createDataFrame([("Amit", 28), ("Sara", 34), ("Ravi", 25)], ["name", "age"])
df.show()

In [0]:
# Write to Unity Catalog table instead of path-based storage
sales_df.write.format("delta").mode("overwrite").saveAsTable("retail.people_demo")

# Create a streaming DataFrame (rate source generates test data)
streaming_df = spark.readStream.format("rate").load()
print(f"Created streaming DataFrame with schema: {streaming_df.schema}") 

In [0]:
df = spark.table("retail.bronze_sales")
df.printSchema()
df.show(5)

In [0]:
# SparkContext and RDD operations are not available on Spark Connect (serverless/standard compute)
# Use DataFrame operations instead
data = [1, 2, 3, 4]
result = [x * 2 for x in data]
print(result)

df = spark.range(4).withColumnRenamed("id", "num")
df.selectExpr("num * 2 as doubled").show()  

In [0]:
df = spark.table("retail.bronze_sales")
filtered = df.filter(df.amount > 250)     
print(filtered.count())                   
filtered.show(5)

In [0]:
df = spark.table("retail.bronze_sales")
step1 = df.filter(df.amount > 100)    
step2 = step1.withColumn("tax", step1.amount * 0.18)
step2.show(5)   

In [0]:
# RDD operations are not available on Spark Connect (serverless/standard compute)
# Repartition DataFrame using DataFrame API
df = spark.table("retail.bronze_sales")
df2 = df.repartition(8)
print(f"Repartitioned DataFrame to 8 partitions")

In [0]:
df = spark.table("retail.bronze_sales")
df.groupBy("region").count().show()    
df.orderBy("amount").show(5) 

In [0]:
df = spark.table("retail.bronze_sales")
df.groupBy("region").count().orderBy(F.desc("count")).show() 

df_salted = df.withColumn("salt", (F.rand() * 10).cast("int"))  
df_salted.groupBy("region", "salt").count().show(5)

In [0]:
df = spark.table("retail.bronze_sales")
df.groupBy("region").count().orderBy(F.desc("count")).show()  

df_salted = df.withColumn("salt", (F.rand() * 10).cast("int")) 
df_salted.groupBy("region", "salt").count().show(5)

In [0]:
df = spark.table("retail.bronze_sales").filter("amount > 200").select("order_id", "amount")
df.explain(True)  

In [0]:
# SparkContext.parallelize() is not available on Spark Connect
# Use DataFrame operations instead
df = spark.createDataFrame([(10,), (20,), (30,)], ["value"])
print(f"Sum: {df.agg({'value': 'sum'}).collect()[0][0]}")

spark.sql("SELECT SUM(amount) FROM retail.bronze_sales").show()

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def tier(amount):
    return "High" if amount > 250 else "Low"

tier_udf = udf(tier, StringType())
spark.table("retail.bronze_sales").withColumn("spend_tier", tier_udf(F.col("amount"))).show(5)

In [0]:
from pyspark.sql.functions import broadcast

result = spark.table("retail.bronze_sales").join(broadcast(category_df), "product_category")
result.select("order_id", "product_category", "notes", "amount").show(5)

In [0]:
# Read from Unity Catalog table instead of path-based storage
delta_df = spark.table("retail.bronze_sales")
delta_df.show(5)

In [0]:
# Direct file system access to Delta logs is restricted on serverless compute
# Use Delta table API to inspect table history and metadata instead
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "retail.bronze_sales")
display(dt.history()) 

In [0]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "retail.bronze_sales")
dt.update(condition="order_id = 1", set={"amount": "999.99"})   # atomic update
dt.toDF().filter("order_id = 1").show()

In [0]:
dt = DeltaTable.forName(spark, "retail.bronze_sales")
dt.history().select("version", "operation", "timestamp").show()

In [0]:
bad_df = spark.createDataFrame([(9999, "extra_column_value")], ["order_id", "unexpected_col"])
try:
    bad_df.write.format("delta").mode("append").saveAsTable("retail.bronze_sales")
except Exception as e:
    print("Schema mismatch rejected:", str(e)[:200])

In [0]:
new_col_df = spark.table("retail.bronze_sales").limit(1) \
    .withColumn("channel", F.lit("online")).select("order_id", "channel")

new_col_df.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("retail.bronze_sales")

spark.table("retail.bronze_sales").printSchema()

In [0]:
spark.sql("SELECT * FROM retail.bronze_sales VERSION AS OF 0 WHERE order_id = 1").show()

In [0]:
from delta.tables import DeltaTable

updates = spark.createDataFrame(
    [(1, "North", "Electronics", 777.00, "2026-09-01"),
     (9998, "South", "Grocery", 45.00, "2026-09-05")],
    ["order_id", "region", "product_category", "amount", "order_date"])

target = DeltaTable.forName(spark, "retail.bronze_sales")
target.alias("t").merge(
    updates.alias("s"), "t.order_id = s.order_id"
).whenMatchedUpdate(set={"amount": "s.amount"}) \
 .whenNotMatchedInsert(values={
     "order_id": "s.order_id", "region": "s.region",
     "product_category": "s.product_category",
     "amount": "s.amount", "order_date": "s.order_date"
 }).execute()

spark.sql("SELECT * FROM retail.bronze_sales WHERE order_id IN (1, 9998)").show()

In [0]:
%sql
OPTIMIZE retail.bronze_sales ZORDER BY (region);

In [0]:
%sql
VACUUM retail.bronze_sales RETAIN 168 HOURS;

In [0]:
spark.sql("SELECT region, SUM(amount) FROM retail.bronze_sales GROUP BY region").show()  # BI
spark.read.table("retail.bronze_sales").select("amount")  

In [0]:
spark.sql("DESCRIBE DETAIL retail.bronze_sales").show()

In [0]:
silver_df = (spark.table("retail.bronze_sales")
    .dropDuplicates(["order_id"])
    .filter("amount > 0")
    .withColumn("order_date", F.to_date("order_date")))
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail.silver_sales")

gold_df = (spark.table("retail.silver_sales")
    .groupBy("region", "product_category")
    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders")))
gold_df.write.format("delta").mode("overwrite").saveAsTable("retail.gold_sales_summary")

In [0]:
spark.sql("SELECT * FROM retail.gold_sales_summary ORDER BY total_revenue DESC").show()

In [0]:

spark.sql("DESCRIBE DETAIL retail.bronze_sales").show()


spark.sql(f"""
    CREATE TABLE retail.bronze_sales_external
    USING DELTA
    LOCATION '{base_path}/bronze_sales_external'
    AS SELECT * FROM retail.bronze_sales
""")
spark.sql("DESCRIBE DETAIL retail.bronze_sales_external").show()

In [0]:
%sql
CREATE OR REPLACE VIEW retail.high_value_orders AS
SELECT * FROM retail.bronze_sales WHERE amount > 300;

SELECT * FROM retail.high_value_orders LIMIT 5;

In [0]:
%sql
SELECT region, SUM(total_revenue) AS region_total
FROM retail.gold_sales_summary
GROUP BY region
ORDER BY region_total DESC;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.retail;
CREATE TABLE main.retail.orders AS SELECT * FROM retail.bronze_sales;
SELECT * FROM main.retail.orders LIMIT 5;

In [0]:
%sql
GRANT SELECT ON TABLE main.retail.orders TO `analyst_group`;
REVOKE SELECT ON TABLE main.retail.orders FROM `intern_group`;

In [0]:
%sql
SHOW DATABASES;                
SHOW TABLES IN retail;


In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", f"{base_path}/schema/retail_orders")
      .load("/Volumes/<catalog>/<schema>/landing/retail_orders/"))   # replace with a real landing path

query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", f"{base_path}/checkpoint/retail_orders")
         .table("retail.bronze_sales_stream"))

In [0]:
rate_stream = spark.readStream.format("rate").load()  
counts = rate_stream.groupBy().count()

query = (counts.writeStream
         .outputMode("complete")
         .format("console")
         .trigger(processingTime="5 seconds")
         .start())


query.stop()

In [0]:
rate_stream = spark.readStream.format("rate").load()

query = (rate_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{base_path}/checkpoints/rate_demo")   # tracks progress for restart
    .outputMode("append")
    .table("retail.rate_stream_demo"))

query.stop()

In [0]:
from delta.tables import DeltaTable

spark.sql("ALTER TABLE retail.bronze_sales SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Make a change so there's something to capture
DeltaTable.forName(spark, "retail.bronze_sales").update(condition="order_id = 2", set={"amount": "1.00"})

changes = (spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .table("retail.bronze_sales"))
changes.select("order_id", "amount", "_change_type", "_commit_version").show()

In [0]:
# Step 1: Bronze — raw ingest (already created in SETUP)
spark.table("retail.bronze_sales").count()

# Step 2: Silver — clean & dedupe
silver_df = (spark.table("retail.bronze_sales")
    .dropDuplicates(["order_id"])
    .filter("amount > 0"))
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail.silver_sales")

# Step 3: Gold — aggregate for BI
gold_df = (spark.table("retail.silver_sales")
    .groupBy("region", "product_category")
    .agg(F.sum("amount").alias("total_revenue")))
gold_df.write.format("delta").mode("overwrite").saveAsTable("retail.gold_sales_summary")

# Step 4: Optimize for query performance
spark.sql("OPTIMIZE retail.gold_sales_summary ZORDER BY (region)")

# Step 5: Verify
spark.sql("SELECT * FROM retail.gold_sales_summary ORDER BY total_revenue DESC").show()